# 04 — Model

The single, growing model notebook. Same temporal train/test split throughout (train up to mid-2024, test after) so every version is comparable to every earlier one. One `fit_and_evaluate()` helper is reused for each version — adding a new feature here after `03_features.ipynb` grows means adding one line, not copy-pasting a new fitting cell.

**Reading the metrics, in plain terms:**
- **Test R²** — how much of the price variation the model explains on sales it never saw during training. 0 = no better than guessing the average; 1 = perfect.
- **MdAPE** (median absolute % error) — typical error size. If MdAPE is 15%, half of predictions were off by less than 15% of the true price, half by more.
- **PPE10** — the fraction of predictions that landed within 10% of the true price. The industry standard way to grade an automated valuation model.

In [1]:
import sys
sys.path.insert(0, "..")
from src.config import DATA_INTERIM
import pandas as pd
import numpy as np
import statsmodels.api as sm

df = pd.read_parquet(DATA_INTERIM / "sefton_features.parquet")
df["log_price"] = np.log(df["price"])
df["log_price_adj"] = np.log(df["price_adjusted"])
df["log_area"] = np.log(df["total_floor_area"])
df["log_relsize"] = np.log(df["relative_size"])
df["dist_to_beach_km"] = df["dist_to_beach_m"] / 1000

split_date = pd.Timestamp("2024-07-01")
print(f"{len(df):,} rows loaded")

61,310 rows loaded


## The reusable fit-and-evaluate helper

Takes a target column (raw price or inflation-adjusted price) and a list of feature columns, fits on the training split, scores on the test split, and prints the same set of metrics every time so versions are always directly comparable.

In [2]:
def fit_and_evaluate(feature_cols, label, target_col="log_price_adj"):
    data = df.dropna(subset=feature_cols + [target_col, "date_of_transfer"])
    train = data[data["date_of_transfer"] < split_date]
    test = data[data["date_of_transfer"] >= split_date]

    X_train = sm.add_constant(train[feature_cols])
    model = sm.OLS(train[target_col], X_train).fit()

    X_test = sm.add_constant(test[feature_cols])
    pred_log = model.predict(X_test)
    actual = np.exp(test[target_col])
    pred = np.exp(pred_log)

    ape = (pred - actual).abs() / actual
    mdape = ape.median()
    ppe10 = (ape <= 0.10).mean()
    ss_res = ((test[target_col] - pred_log) ** 2).sum()
    ss_tot = ((test[target_col] - test[target_col].mean()) ** 2).sum()
    test_r2 = 1 - ss_res / ss_tot

    print(f"--- {label} ---")
    print(f"n = {len(train):,} train / {len(test):,} test")
    print(f"Train R2:  {model.rsquared:.3f}")
    print(f"Test R2:   {test_r2:.3f}")
    print(f"Test MdAPE: {mdape:.1%}")
    print(f"Test PPE10: {ppe10:.1%}")
    print()
    return model, pred, {"mdape": mdape, "ppe10": ppe10, "test_r2": test_r2}


## Model versions, in order built

- **v0 — Phase 1 baseline**: floor area only, *no* inflation adjustment. Kept as the fixed reference point everything else is measured against (test R² was ≈0 here — this is what exposed the need for the fix below).
- **v1**: same, but inflation-adjusted. Isolates how much that one fix was worth.
- **v2**: adds `relative_size` (house size vs its 15 nearest neighbours).
- **v3**: adds house type and coastal distance.

In [3]:
built_form_dummies = [c for c in df.columns if c.startswith("type_")]

m0, p0, s0 = fit_and_evaluate(["log_area"], "v0: floor area only (NOT inflation-adjusted)", target_col="log_price")
m1, p1, s1 = fit_and_evaluate(["log_area"], "v1: floor area only (inflation-adjusted)")
m2, p2, s2 = fit_and_evaluate(["log_area", "log_relsize"], "v2: + relative_size")
m3, p3, s3 = fit_and_evaluate(["log_area", "log_relsize", "dist_to_beach_km"] + built_form_dummies, "v3: + house type + coastal distance")

--- v0: floor area only (NOT inflation-adjusted) ---
n = 46,789 train / 6,139 test
Train R2:  0.452
Test R2:   -0.009
Test MdAPE: 33.6%
Test PPE10: 10.0%

--- v1: floor area only (inflation-adjusted) ---
n = 46,789 train / 6,139 test
Train R2:  0.482
Test R2:   0.492
Test MdAPE: 20.1%
Test PPE10: 25.5%



--- v2: + relative_size ---
n = 46,789 train / 6,139 test
Train R2:  0.542
Test R2:   0.560
Test MdAPE: 18.0%
Test PPE10: 28.4%



--- v3: + house type + coastal distance ---
n = 46,789 train / 6,139 test
Train R2:  0.656
Test R2:   0.668
Test MdAPE: 15.1%
Test PPE10: 34.7%



## Progress so far

One table, every version, so the next feature added to `03_features.ipynb` has an obvious bar to clear.

In [4]:
summary = pd.DataFrame({
    "version": ["v0: baseline (no HPI fix)", "v1: + HPI adjustment", "v2: + relative_size", "v3: + house type + coastal"],
    "test_r2": [s0["test_r2"], s1["test_r2"], s2["test_r2"], s3["test_r2"]],
    "mdape": [s0["mdape"], s1["mdape"], s2["mdape"], s3["mdape"]],
    "ppe10": [s0["ppe10"], s1["ppe10"], s2["ppe10"], s3["ppe10"]],
}).set_index("version")
print(summary.to_string(formatters={"mdape": "{:.1%}".format, "ppe10": "{:.1%}".format, "test_r2": "{:.3f}".format}))

                           test_r2 mdape ppe10
version                                       
v0: baseline (no HPI fix)   -0.009 33.6% 10.0%
v1: + HPI adjustment         0.492 20.1% 25.5%
v2: + relative_size          0.560 18.0% 28.4%
v3: + house type + coastal   0.668 15.1% 34.7%


## What v3 is actually basing predictions on

OLS coefficients on a log-log model translate to "percent change in price per unit change in the feature" — useful for sanity-checking that the model has learned something sensible, not just a black box.

In [5]:
print(m3.summary())

                            OLS Regression Results                            
Dep. Variable:          log_price_adj   R-squared:                       0.656
Model:                            OLS   Adj. R-squared:                  0.656
Method:                 Least Squares   F-statistic:                 1.274e+04
Date:                Sat, 08 Aug 2026   Prob (F-statistic):               0.00
Time:                        10:49:22   Log-Likelihood:                -12108.
No. Observations:               46789   AIC:                         2.423e+04
Df Residuals:                   46781   BIC:                         2.430e+04
Df Model:                           7                                         
Covariance Type:            nonrobust                                         
                         coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------
const                  8.1189      0

## Stratified check: does relative_size help most where floor area alone struggles?

The original hypothesis (plan Part 2): on uniform-stock streets, plain floor area stops predicting price at all. `relative_size` was meant to rescue exactly that case. Comparing v1 (no relative_size) against v2 (with it), split by how much floor-area variety each area has.

In [6]:
eval_df = df.dropna(subset=["log_area", "log_relsize", "log_price_adj", "date_of_transfer"])
test_eval = eval_df[eval_df["date_of_transfer"] >= split_date].copy()
test_eval["postcode_district"] = test_eval["postcode"].str.split(" ").str[0]

area_variance = eval_df.assign(postcode_district=eval_df["postcode"].str.split(" ").str[0]) \
    .groupby("postcode_district")["total_floor_area"].std()
test_eval["area_variance_tier"] = pd.qcut(
    test_eval["postcode_district"].map(area_variance), q=3, labels=["low", "medium", "high"]
)

test_eval["pred_v1"] = p1.reindex(test_eval.index)
test_eval["pred_v2"] = p2.reindex(test_eval.index)
test_eval["ape_v1"] = (test_eval["pred_v1"] - test_eval["price_adjusted"]).abs() / test_eval["price_adjusted"]
test_eval["ape_v2"] = (test_eval["pred_v2"] - test_eval["price_adjusted"]).abs() / test_eval["price_adjusted"]

stratified = test_eval.groupby("area_variance_tier", observed=True).agg(
    n=("ape_v1", "size"), v1_mdape=("ape_v1", "median"), v2_mdape=("ape_v2", "median"),
)
stratified["improvement"] = stratified["v1_mdape"] - stratified["v2_mdape"]
print(stratified)

                       n  v1_mdape  v2_mdape  improvement
area_variance_tier                                       
low                 2178  0.206028  0.189699     0.016329
medium              2009  0.185331  0.167303     0.018028
high                1952  0.212158  0.183910     0.028247
